In [2]:
import awkward as ak
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional

# A simple logger/named transform input for the example
import logging

logger = logging.getLogger()

from lmetk.utils.awkward_utils import (
    ak_find_var_dims,
    ak_regularize_regular_axes,
    ragged_index_tuples,
)

from lmetk.transforms.base.multisource import find_singular_ragged_dim


class AdaptiveTimeSeriesSequencer:
    """
    Creates time-series windows from a single ragged Awkward Array.
    """

    def __init__(
        self,
        features: ak.Array,
        seq_len: int,
        label_len: int,
        pred_len: int,
        pad: bool = False,
        pad_value: float = 0.0,
    ):

        self.features = features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.pad = pad
        self.pad_value = pad_value

        self.ragged_dim = find_singular_ragged_dim({"features": self.features})
        if self.ragged_dim is None:
            raise ValueError("No ragged dimension found in the input array.")

        if pad:
            raise NotImplementedError(
                "Padding functionality is not checked yet."
                "Please verify correctness before using in production."
                "The masking functionality is not implemented yet."
            )

        example_index = (0,) * (self.ragged_dim + 1)
        example_element = self.features[example_index]
        if isinstance(example_element, (int, float, np.number)):
            self.num_features = 1
        else:
            self.num_features = len(example_element)

        self._indices = self._build_indices()

    def _build_indices(self) -> list[tuple]:
        """
        Build a master list of indices.

        Logic:
        1) Drop samples that are too short to contain a single window.
        2) For each remaining sample, keep only those *starting* timesteps
            that can generate a complete (seq_x, seq_y) window.
        """
        num_indexing_dims = self.ragged_dim + 1

        # 1) Drop sequences that are too short
        lengths = ak.num(self.features, axis=self.ragged_dim)
        min_required = self.seq_len + self.pred_len
        keep_mask = lengths >= min_required
        features_kept = self.features[keep_mask]
        lengths_kept = lengths[keep_mask]

        # 2) For each sample, retain only valid *starting* indices
        # Valid starts: t ∈ [0, t_max] where t_max = L - (seq_len + pred_len)
        trimmed_samples = []
        for sample, L in zip(features_kept, lengths_kept):
            t_max = L - (self.seq_len + self.pred_len)
            # Keep just the range of starting indices 0..t_max inclusive
            # (ragged_index_tuples enumerates positions along this axis)
            cut_len = max(0, t_max + 1)
            trimmed_samples.append(sample[:cut_len])

        trimmed_array = ak.Array(trimmed_samples)

        # 3) Enumerate all valid (sample_idx, t, ...) tuples
        all_possible_indices = ragged_index_tuples(trimmed_array, num_indexing_dims)
        return all_possible_indices

    def __len__(self) -> int:
        return len(self._indices)

    def _pad_and_fill(self, array: ak.Array, target_len: int) -> ak.Array:
        """Pads a sliced array to the target length."""
        if len(array) == 0:
            shape = (target_len, self.num_features)
            return ak.from_numpy(np.full(shape, self.pad_value, dtype=np.float32))

        padded = ak.pad_none(array, target_len, axis=0)
        builder = ak.ArrayBuilder()
        pad_record = np.full(self.num_features, self.pad_value, dtype=np.float32)
        for record in padded:
            # Here, we must ensure scalars are appended in a way that matches the pad_record structure
            if record is None:
                builder.append(pad_record)
            elif self.num_features == 1 and isinstance(record, (int, float, np.number)):
                builder.append(np.array([record], dtype=np.float32))  # Wrap scalar
            else:
                builder.append(record)
        return builder.snapshot()

    def _ak_to_tensor(self, array: ak.Array) -> torch.Tensor:
        """Converts a finalized Awkward array to a PyTorch tensor."""
        np_arr = ak.to_numpy(array)
        if self.num_features == 1 and np_arr.ndim == 1:
            np_arr = np_arr.reshape(-1, 1)
        return torch.from_numpy(np_arr).to(torch.float32)

    def get_window_for_index(
        self, index_tuple: tuple
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Extracts, pads (if enabled), and converts a single window."""
        sample_idx = index_tuple[: self.ragged_dim]
        t = index_tuple[self.ragged_dim]
        sample = self.features[sample_idx]

        x_start, x_end = t, t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len

        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]

        # --- KEY FIX IS HERE ---
        # Only call the padding function when self.pad is True.
        if self.pad:
            seq_x_ak = self._pad_and_fill(seq_x_ak, self.seq_len)
            seq_y_ak = self._pad_and_fill(seq_y_ak, self.label_len + self.pred_len)

        return self._ak_to_tensor(seq_x_ak), self._ak_to_tensor(seq_y_ak)


class SequenceDataset(Dataset):
    """Standard PyTorch Dataset wrapper."""

    def __init__(self, sequencer: AdaptiveTimeSeriesSequencer):
        self.seq = sequencer

    def __len__(self) -> int:
        return len(self.seq)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        index_tuple = self.seq._indices[idx]
        return self.seq.get_window_for_index(index_tuple)


if __name__ == "__main__":

    features_complex = ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]])
    print("PADDING is set to FALSE")
    # The sequencer automatically detects ragged_dim = 2
    sequencer_complex = AdaptiveTimeSeriesSequencer(
        features=features_complex,
        seq_len=3,
        label_len=1,
        pred_len=1,
        pad=False,
        pad_value=-1,
    )

    dataset_complex = SequenceDataset(sequencer_complex)
    # The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
    for i in range(dataset_complex.__len__()):
        seq_x, seq_y = dataset_complex[i]

        print("\n## Complex Data (N * var )")
        print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
        print("seq_x:\n", seq_x)
        print("seq_y:\n", seq_y)

    
    print("PADDING is set to FALSE")
    features_complex = ak_regularize_regular_axes(
        ak.Array(
            [
                [[0, 1], [2, 3]],  # Sample 0  # Sub-sample (0,0)
                [[10, 11]],  # Sample 1  # Sub-sample (1,0)
                [[4, 5], [6, 7], [8, 9]],
            ]
        )
    )
    # The sequencer automatically detects ragged_dim = 2
    sequencer_complex = AdaptiveTimeSeriesSequencer(
        features=features_complex,
        seq_len=2,
        label_len=1,
        pred_len=1,
        pad=False,
        pad_value=-1,
    )

    dataset_complex = SequenceDataset(sequencer_complex)
    # The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
    for i in range(dataset_complex.__len__()):
        seq_x, seq_y = dataset_complex[i]

        print("\n## Complex Data (N * var * C)")
        print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
        print("seq_x:\n", seq_x)
        print("seq_y:\n", seq_y)

    

PADDING is set to FALSE

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[0.],
        [1.],
        [3.]])
seq_y:
 tensor([[3.],
        [4.]])

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[5.],
        [6.],
        [7.]])
seq_y:
 tensor([[7.]])

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[6.],
        [7.]])
seq_y:
 tensor([], size=(0, 1))

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[7.]])
seq_y:
 tensor([], size=(0, 1))

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([], size=(0, 1))
seq_y:
 tensor([], size=(0, 1))
PADDING is set to FALSE

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[0., 1.],
        [2., 3.]])
seq_y:
 tensor([[2., 3.]])


In [3]:
import awkward as ak
import numpy as np
import torch
from typing import List, Optional, Tuple
from torch.utils.data import Dataset

# ----------------------------------------------------------------------
# AkwardJaggedAutoregressiveSequencer
# ----------------------------------------------------------------------
class AkwardJaggedAutoregressiveSequencer:
    """
    Autoregressive sequencer for *ragged* Awkward Arrays.

    Creates overlapping input/output pairs from a single ragged Awkward Array,
    analogous to AutoregressiveSequencer but operating on jagged segments.

    Args:
        features: Awkward Array of shape (N, variable_length, D) or deeper
        time_features: optional Awkward Array with the same ragged structure
        seq_len: Length of the encoder input window
        label_len: Overlap between encoder and decoder
        pred_len: Length of the prediction horizon
        stride: Step size between consecutive windows
        pad: Whether to pad sequences shorter than required (default False)
        pad_value: Padding value if pad=True
    """

    def __init__(
        self,
        features: ak.Array,
        time_features: Optional[ak.Array],
        seq_len: int,
        label_len: int,
        pred_len: int,
        stride: int = 1,
        pad: bool = False,
        pad_value: float = 0.0,
    ):
        self.features = features
        self.time_features = time_features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.stride = stride
        self.pad = pad
        self.pad_value = pad_value
        self.device = torch.device("cpu")
        
        assert features.ndim < 4, "Features dim greater than 3 has not been tested"
        # Determine which axis is ragged (same helper logic as AdaptiveTimeSeriesSequencer)
        from lmetk.transforms.base.multisource import find_singular_ragged_dim
        self.ragged_dim = find_singular_ragged_dim({"features": self.features})
        if self.ragged_dim is None:
            raise ValueError("No ragged dimension found in the input array.")

        # Count feature dimension
        example_index = (0,) * (self.ragged_dim + 1)
        example_element = self.features[example_index]
        if isinstance(example_element, (int, float, np.number)):
            self.num_features = 1
        else:
            self.num_features = len(example_element)

        # Precompute valid window bounds: (sample_idx..., start, end, r_start, p_end)
        self._indices = self._build_indices()

    def _build_indices(self) -> List[tuple]:
        from lmetk.utils.awkward_utils import ragged_index_tuples
        num_indexing_dims = self.ragged_dim + 1
        all_possible = ragged_index_tuples(self.features, num_indexing_dims)

        valid = []
        for index_tuple in all_possible:
            # index_tuple = (...sample dims..., t)
            sample_idx = index_tuple[: self.ragged_dim]
            t = index_tuple[self.ragged_dim]
            sample_len = len(self.features[sample_idx])
            # Compute required length for this start t
            required_end = (t + self.seq_len - self.label_len) + (self.label_len + self.pred_len)
            if required_end <= sample_len:
                # enforce stride: only keep starts that respect stride
                if (t % self.stride) == 0:
                    valid.append(index_tuple)
        return valid

    def __len__(self) -> int:
        return len(self._indices)

    # ------------------------------------------------------------------
    # Core helpers
    # ------------------------------------------------------------------
    def _ak_to_tensor(self, array: ak.Array) -> torch.Tensor:
        np_arr = ak.to_numpy(array)
        if self.num_features == 1 and np_arr.ndim == 1:
            np_arr = np_arr.reshape(-1, 1)
        return torch.from_numpy(np_arr).to(torch.float32)

    def _pad_and_fill(self, array: ak.Array, target_len: int) -> ak.Array:
        if len(array) == 0:
            shape = (target_len, self.num_features)
            return ak.from_numpy(np.full(shape, self.pad_value, dtype=np.float32))

        padded = ak.pad_none(array, target_len, axis=0)
        builder = ak.ArrayBuilder()
        pad_record = np.full(self.num_features, self.pad_value, dtype=np.float32)
        for record in padded:
            if record is None:
                builder.append(pad_record)
            elif self.num_features == 1 and isinstance(record, (int, float, np.number)):
                builder.append(np.array([record], dtype=np.float32))
            else:
                builder.append(record)
        return builder.snapshot()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def get_window_for_index(
        self, index_tuple: tuple
    ) -> Tuple[torch.Tensor, torch.Tensor,
               Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Extract a single encoder/decoder pair.

        Returns:
            seq_x : (seq_len, D) tensor
            seq_y : (label_len + pred_len, D) tensor
            seq_x_time : optional time features
            seq_y_time : optional time features
        """
        sample_idx = index_tuple[: self.ragged_dim]
        t = index_tuple[self.ragged_dim]
        sample = self.features[sample_idx]

        x_start, x_end = t, t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len

        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]

        if self.pad:
            seq_x_ak = self._pad_and_fill(seq_x_ak, self.seq_len)
            seq_y_ak = self._pad_and_fill(seq_y_ak, self.label_len + self.pred_len)

        seq_x = self._ak_to_tensor(seq_x_ak)
        seq_y = self._ak_to_tensor(seq_y_ak)

        if self.time_features is not None:
            tf_sample = self.time_features[sample_idx]
            seq_x_time = self._ak_to_tensor(tf_sample[x_start:x_end])
            seq_y_time = self._ak_to_tensor(tf_sample[y_start:y_end])
        else:
            seq_x_time = None
            seq_y_time = None

        return seq_x, seq_y, seq_x_time, seq_y_time


# ----------------------------------------------------------------------
# Simple Dataset wrapper
# ----------------------------------------------------------------------
class JaggedSequenceDataset(Dataset):
    def __init__(self, sequencer: AkwardJaggedAutoregressiveSequencer):
        self.seq = sequencer

    def __len__(self) -> int:
        return len(self.seq)

    def __getitem__(self, idx: int):
        index_tuple = self.seq._indices[idx]
        return self.seq.get_window_for_index(index_tuple)
    



#features = ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]])
features = ak_regularize_regular_axes(
        ak.Array(
            [
                [[0, 1], [2, 3]],  # Sample 0  # Sub-sample (0,0)
                [[10, 11]],  # Sample 1  # Sub-sample (1,0)
                [[4, 5], [6, 7], [8, 9],[10, 11], [12, 13]],
            ]
        )
    )
time_feats = None

seq = AkwardJaggedAutoregressiveSequencer(
    features=features,
    time_features=time_feats,
    seq_len=2,
    label_len=0,
    pred_len=1,
    stride=2,
)

dataset = JaggedSequenceDataset(seq)

# The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
for i in range(dataset.__len__()):
    seq_x, seq_y, _, _ = dataset[i]

    print("\n## Complex Data (N * var * C)")
    print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
    print("seq_x:\n", seq_x)
    print("seq_y:\n", seq_y)





## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[4., 5.],
        [6., 7.]])
seq_y:
 tensor([[8., 9.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[ 8.,  9.],
        [10., 11.]])
seq_y:
 tensor([[12., 13.]])


ArrayType(ListType(RegularType(NumpyType('int64'), 2)), 3, None)

In [34]:
from typing import Optional, List, Dict, Union
import numpy as np
import torch
import awkward as ak
from torch.utils.data import Dataset
from lmetk.data.optimization.sequencer import AutoregressiveSequencer
from lmetk.data.optimization.ak_jagged_sequencer import AkwardJaggedAutoregressiveSequencer
ArrayLike = Union[np.ndarray, torch.Tensor, ak.Array]


# ----------------------------------------------------------------------
# Base drivers
# ----------------------------------------------------------------------
class BaseSequencer:
    def __len__(self): ...
    def sequences_batch(self, index_subset): ...


class DenseDriver(BaseSequencer):
    def __init__(self, array: Union[np.ndarray, torch.Tensor],
                 seq_len: int, label_len: int, pred_len: int, stride: int):
        if isinstance(array, np.ndarray):
            array = torch.from_numpy(array).float()
        self.driver = AutoregressiveSequencer(
            features=array, time_features=None,  # no time features by default
            seq_len=seq_len, label_len=label_len,
            pred_len=pred_len, stride=stride
        )

    def __len__(self): return len(self.driver)
    def sequences_batch(self, idx): return self.driver.sequences_batch(idx)


class RaggedDriver(BaseSequencer):
    def __init__(self, array: ak.Array,
                 seq_len: int, label_len: int, pred_len: int,
                 stride: int, pad: bool = False, pad_value: float = 0.0):
        self.driver = AkwardJaggedAutoregressiveSequencer(
            features=array, time_features=None,  # no time features by default
            seq_len=seq_len, label_len=label_len,
            pred_len=pred_len, stride=stride,
            pad=pad, pad_value=pad_value
        )

    def __len__(self): return len(self.driver)
    def sequences_batch(self, idx): return self.driver.sequences_batch(idx)


# ----------------------------------------------------------------------
# Unified dataset
# ----------------------------------------------------------------------
class SlidingWindowBatchDataset(Dataset):
    """
    Create a separate sequencer for every key in `data_dict`.
    Time features are NOT required.
    """
    def __init__(self,
                 data_dict: Dict[str, ArrayLike],
                 seq_len: int,
                 label_len: int,
                 pred_len: int,
                 stride: int,
                 pad: bool = False,
                 pad_value: float = 0.0):

        self.sequencers: Dict[str, BaseSequencer] = {}
        self._length: Optional[int] = None

        for key, arr in data_dict.items():
            if arr is None:
                continue

            if isinstance(arr, (np.ndarray, torch.Tensor)):
                driver = DenseDriver(arr, seq_len, label_len, pred_len, stride)
            elif isinstance(arr, ak.Array):
                driver = RaggedDriver(arr, seq_len, label_len,
                                      pred_len, stride, pad, pad_value)
            else:
                raise NotImplementedError(f"Unsupported type for key '{key}': {type(arr)}")

            self.sequencers[key] = driver

            # ensure all sequencers have the same length
            if self._length is None:
                self._length = len(driver)
            elif len(driver) != self._length:
                raise ValueError(f"All sequences must yield the same number of windows "
                                 f"(mismatch at key '{key}')")

    def __len__(self):
        return self._length or 0

    def __getitem__(self, idx: Union[int, List[int]]) -> Dict[str, tuple]:
        """
        Returns a dict of {key: (x, y, x_time, y_time)}.
        For drivers without time features the last two entries are None.
        """
        out = {}
        for k, drv in self.sequencers.items():
            x, y, x_time, y_time = drv.sequences_batch(idx)
            out[k] = (x, y, x_time, y_time)
        return out

    def get_collate_fn(self):
        from lmetk.data.datasets.collate_functions import forecasting_collate_fn
        return forecasting_collate_fn


In [41]:
import numpy as np, torch, awkward as ak

data = {
    "features":ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]]),
    "time_stamps":   ak.Array([[0, 0, 0, 0], [1, 1, 1], [2, 2, 2, 2, 2, 2, 2]]),
    
}

dataset = SlidingWindowBatchDataset(
    data_dict=data,
    seq_len=5, label_len=0, pred_len=2, stride=3
)

batch = dataset[0]
for key, val in batch.items():
    print("key:", key) 
    
    seq_x, seq_y, _, _ = val
    print("seq_x.shape:", seq_x.shape)  
    print("seq_x:", seq_x)

    print("seq_y.shape:", seq_y.shape)  
    print("seq_y:", seq_y)  

    print('-------')
    

key: features
seq_x.shape: torch.Size([1, 5, 1])
seq_x: tensor([[[ 8.],
         [ 9.],
         [10.],
         [11.],
         [12.]]])
seq_y.shape: torch.Size([1, 2, 1])
seq_y: tensor([[[13.],
         [14.]]])
-------
key: time_stamps
seq_x.shape: torch.Size([1, 5, 1])
seq_x: tensor([[[2.],
         [2.],
         [2.],
         [2.],
         [2.]]])
seq_y.shape: torch.Size([1, 2, 1])
seq_y: tensor([[[2.],
         [2.]]])
-------


In [42]:
batch

{'features': (tensor([[[ 8.],
           [ 9.],
           [10.],
           [11.],
           [12.]]]),
  tensor([[[13.],
           [14.]]]),
  None,
  None),
 'time_stamps': (tensor([[[2.],
           [2.],
           [2.],
           [2.],
           [2.]]]),
  tensor([[[2.],
           [2.]]]),
  None,
  None)}

In [39]:
import numpy as np, torch, awkward as ak

data = {
    "features": np.arange(20).reshape(-1,1),
    "labels":   torch.arange(20).float().view(-1,1),
    "events":   ak.Array([[0,1,2],[3,4],[5,6,7,8]])
}

dataset = SlidingWindowBatchDataset(
    data_dict=data,
    seq_len=5, label_len=0, pred_len=2, stride=3
)

batch = dataset[0]
print(batch.keys())  # dict_keys(['features', 'labels', 'events'])
print(batch["features"][0].shape)  # torch.Size([5,1])
print(batch["events"][0])          # awkward.Array slice


ValueError: All sequences must yield the same number of windows (mismatch at key 'events')

key: features
seq_x.shape: torch.Size([1, 5, 1])
seq_x: tensor([[[ 8.],
         [ 9.],
         [10.],
         [11.],
         [12.]]])
seq_y.shape: torch.Size([1, 2, 1])
seq_y: tensor([[[13.],
         [14.]]])
-------
key: time_features
seq_x.shape: torch.Size([1, 5, 1])
seq_x: tensor([[[2.],
         [2.],
         [2.],
         [2.],
         [2.]]])
seq_y.shape: torch.Size([1, 2, 1])
seq_y: tensor([[[2.],
         [2.]]])
-------


In [ ]:
import numpy as np, torch, awkward as ak

data = {
    "features":ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]]),
    "targets":   ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]])
}

dataset = SlidingWindowBatchDataset(
    data_dict=data,
    seq_len=5, label_len=0, pred_len=2, stride=3
)

batch = dataset[0]
print(batch.keys())  # dict_keys(['features', 'labels', 'events'])
print(batch["features"][0].shape)  # torch.Size([5,1])
print(batch["targets"][0])          # awkward.Array slice